In [13]:
from pathlib import Path


def load_env(name):
    """Reads one variable from the first .env found, going up from the current folder."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        env_file = folder / ".env"
        if env_file.exists():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                key, sep, value = line.partition("=")
                if sep and key.strip() == name:
                    return value.strip().strip("\"'")
    raise KeyError(f"{name} not found in any .env")


In [14]:
import pandas as pd
import boto3
from botocore.exceptions import BotoCoreError, ClientError


def load_env_first(names, default=None):
    """The value of the first of these variables found in .env. Without a default, none found is an error."""
    for name in names:
        try:
            return load_env(name)
        except KeyError:
            continue
    if default is None:
        raise KeyError(f"none of {', '.join(names)} found in any .env")
    return default


def make_aws_session():
    """boto3 session with the AWS credentials from .env. Accepts AWS_ACCESS_KEY_ID or AWS_ACCESS_KEY, AWS_SECRET_ACCESS_KEY or AWS_SECRET_KEY, and AWS_SESSION_TOKEN when the keys are temporary. The region is AWS_DEFAULT_REGION or AWS_REGION, and us-east-1 when neither is set."""
    return boto3.Session(
        aws_access_key_id=load_env_first(["AWS_ACCESS_KEY_ID", "AWS_ACCESS_KEY"]),
        aws_secret_access_key=load_env_first(["AWS_SECRET_ACCESS_KEY", "AWS_SECRET_KEY"]),
        aws_session_token=load_env_first(["AWS_SESSION_TOKEN"], default="") or None,  # only temporary credentials (SSO, assumed roles) have one
        region_name=load_env_first(["AWS_DEFAULT_REGION", "AWS_REGION"], default="us-east-1"),
    )


def test_aws(session):
    """Asks AWS who these credentials belong to: one call that needs no permission, so it only fails when the keys or the region are wrong."""
    try:
        identity = session.client("sts").get_caller_identity()
        print(f"OK [{session.region_name}]: AWS credentials accepted")
        print(f"account: {identity['Account']} | identity: {identity['Arn']}")
    except ClientError as e:
        error = e.response["Error"]
        print(f"ERROR {error['Code']}: {error['Message']}")
    except BotoCoreError as e:
        print(f"Connection ERROR: {e}")


aws_session = make_aws_session()
test_aws(aws_session)

OK [us-east-1]: AWS credentials accepted
account: 123456789012 | identity: arn:aws:iam::123456789012:user/undercover


In [15]:
def aws_error(e):
    """One line for a failed AWS call. AccessDenied means the keys work but this user lacks that permission."""
    error = e.response["Error"]
    return f"{error['Code']}: {error['Message']}"


def check_mfa(session):
    """Says whether the root account has MFA, and whether the identity these keys belong to has an MFA device."""
    identity = session.client("sts").get_caller_identity()
    iam = session.client("iam")
    try:
        root_mfa = iam.get_account_summary()["SummaryMap"].get("AccountMFAEnabled")
        print(f"root account MFA: {'ON' if root_mfa else 'OFF: turn it on before anything else'}")
    except ClientError as e:
        print(f"root account MFA: could not check ({aws_error(e)})")

    resource = identity["Arn"].split(":", 5)[5]  # for example "user/alex", "root" or "assumed-role/Admin/session"
    if resource == "root":
        print("identity: these are the ROOT keys. Create an IAM user or use SSO, and delete the root keys.")
    elif resource.startswith("user/"):
        user = resource.split("/")[-1]
        try:
            devices = iam.list_mfa_devices(UserName=user)["MFADevices"]
            print(f"user {user} MFA: {'ON (' + str(len(devices)) + ' device)' if devices else 'OFF: add a device to this user'}")
        except ClientError as e:
            print(f"user {user} MFA: could not check ({aws_error(e)})")
    else:
        print(f"identity: {resource.split('/')[0]}, not an IAM user, so there is no MFA device to list here. With SSO the MFA is set in IAM Identity Center.")


check_mfa(aws_session)

root account MFA: ON
user undercover MFA: could not check (AccessDenied: User: arn:aws:iam::123456789012:user/undercover is not authorized to perform: iam:ListMFADevices on resource: user undercover because no identity-based policy allows the iam:ListMFADevices action. Go to https://us-east-1.console.aws.amazon.com/iam/home?region=us-east-1#/authorization-details/5e17khvc45ug0qp3opuxe422o for complete details, or call the GetRequestAuthorizationDetails API with the following authorization id: 5e17khvc45ug0qp3opuxe422o)


In [16]:
def check_budgets(session):
    """Lists the budgets of the account: the limit, the spend so far, the forecast and how many alerts each one has."""
    account_id = session.client("sts").get_caller_identity()["Account"]
    budgets_client = session.client("budgets")
    try:
        budgets = [b for page in budgets_client.get_paginator("describe_budgets").paginate(AccountId=account_id) for b in page.get("Budgets", [])]
    except ClientError as e:
        print(f"budgets: could not list them ({aws_error(e)})")
        return None
    if not budgets:
        print("budgets: none. Create one in Billing > Budgets, with an e-mail alert, so a mistake does not turn into a bill.")
        return pd.DataFrame()
    rows = []
    for budget in budgets:
        name = budget["BudgetName"]
        spend = budget.get("CalculatedSpend", {})
        try:
            alerts = len(budgets_client.describe_notifications_for_budget(AccountId=account_id, BudgetName=name).get("Notifications", []))
        except ClientError:
            alerts = None  # no permission to read the alerts
        rows.append({
            "budget": name,
            "limit": float(budget["BudgetLimit"]["Amount"]) if "BudgetLimit" in budget else None,
            "unit": budget.get("BudgetLimit", {}).get("Unit"),
            "period": budget["TimeUnit"],
            "type": budget["BudgetType"],
            "spent": float(spend["ActualSpend"]["Amount"]) if "ActualSpend" in spend else None,
            "forecast": float(spend["ForecastedSpend"]["Amount"]) if "ForecastedSpend" in spend else None,
            "alerts": alerts,
        })
    table = pd.DataFrame(rows)
    print(f"{len(table)} budget(s) in account {account_id}" + ("" if (table["alerts"].fillna(0) > 0).all() else ". A budget with 0 alerts only shows the numbers: nobody gets an e-mail."))
    return table


budgets_table = check_budgets(aws_session)
budgets_table

1 budget(s) in account 123456789012


,budget,limit,unit,period,type,spent,forecast,alerts
0,monthly,5.0,USD,MONTHLY,COST,0.0,None,3


In [17]:
def list_bedrock_models(session, provider=None):
    """What this account can see in Bedrock, in this region: the models and the inference profiles. Newer models are usually called through an inference profile, whose id starts with a region such as us."""
    bedrock = session.client("bedrock")
    try:
        models = bedrock.list_foundation_models(**({"byProvider": provider} if provider else {}))["modelSummaries"]
        profiles = bedrock.list_inference_profiles(maxResults=1000)["inferenceProfileSummaries"]
    except ClientError as e:
        print(f"Bedrock: could not list ({aws_error(e)})")
        return None
    rows = [
        {"id": m["modelId"], "name": m["modelName"], "provider": m["providerName"], "kind": "model", "on_demand": "ON_DEMAND" in m.get("inferenceTypesSupported", [])}
        for m in models
    ]
    rows += [
        {"id": p["inferenceProfileId"], "name": p["inferenceProfileName"], "provider": None, "kind": "inference profile", "on_demand": True}
        for p in profiles
        if provider is None or provider.lower() in p["inferenceProfileId"].lower()
    ]
    print(f"Bedrock [{session.region_name}]: {len(models)} models and {len(profiles)} inference profiles visible")
    return pd.DataFrame(rows)


bedrock_models = list_bedrock_models(aws_session, provider="Anthropic")  # provider=None lists every provider
bedrock_models

Bedrock [us-east-1]: 15 models and 101 inference profiles visible


,id,name,provider,kind,on_demand
0,anthropic.claude-haiku-4-5-20251001-v1:0,Claude Haiku 4.5,Anthropic,model,False
1,anthropic.claude-opus-5-5,Claude Opus 5.5,Anthropic,model,False
2,anthropic.claude-fable-5,Claude Fable 5,Anthropic,model,False
3,anthropic.claude-haiku-5-5,Claude Haiku 5.5,Anthropic,model,False
4,anthropic.claude-sonnet-4-6,Claude Sonnet 4.6,Anthropic,model,False
5,anthropic.claude-opus-4-6-v1,Claude Opus 4.6,Anthropic,model,False
6,anthropic.claude-opus-5,Claude Opus 5,Anthropic,model,False
7,anthropic.claude-opus-4-8,Claude Opus 4.8,Anthropic,model,False
8,anthropic.claude-opus-4-7,Claude Opus 4.7,Anthropic,model,False
9,anthropic.claude-sonnet-4-5-20250929-v1:0,Claude Sonnet 4.5,Anthropic,model,False


In [18]:
BEDROCK_MODEL_ID = None  # paste an id from the table above, for example one that starts with us.anthropic


def ask_bedrock(session, model_id, prompt, max_tokens=200):
    """One question to a Bedrock model through the Converse API, which has the same shape for every model."""
    response = session.client("bedrock-runtime").converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens},
    )
    blocks = response["output"]["message"]["content"]
    return "".join(block["text"] for block in blocks if "text" in block), response["usage"]


def test_bedrock(session, model_id):
    """Sends a tiny prompt to the model, which costs a fraction of a cent, and says what went wrong if it fails."""
    if not model_id:
        print("Set BEDROCK_MODEL_ID to an id from the table above.")
        return
    try:
        text, usage = ask_bedrock(session, model_id, "Say hi in five words.")
        print(f"OK [{session.region_name}] {model_id}: {text.strip()}")
        print(f"tokens: {usage['inputTokens']} in, {usage['outputTokens']} out")
    except ClientError as e:
        code = e.response["Error"]["Code"]
        hints = {
            "AccessDeniedException": "this user may lack bedrock:InvokeModel, or the model is not enabled for the account yet (Anthropic models can ask for a one-time use case form in the Bedrock console)",
            "ValidationException": "the id may be wrong, or this model has to be called through an inference profile id (the ones that start with a region)",
            "ResourceNotFoundException": "the model is not in this region",
            "ThrottlingException": "too many calls: wait a moment and retry",
        }
        print(f"ERROR {aws_error(e)}")
        if code in hints:
            print(f"hint: {hints[code]}")
    except BotoCoreError as e:
        print(f"Connection ERROR: {e}")


test_bedrock(aws_session, BEDROCK_MODEL_ID)

Set BEDROCK_MODEL_ID to an id from the table above.
